# Task 4 — YOLOv8 5-fold + masked pool (WBF 앙상블용 체크포인트 생성, Colab)

**목적**: task2-masked와 동일한 데이터(원본 train 232장 + 합성 pool2 + **masked pool**, 74종)로 **YOLOv8**을
5-fold 학습해, 추후 RF-DETR(task2-masked)와 fold-matched WBF 앙상블에 쓸 체크포인트 5개를 만든다.
**classification loss 가중치를 높여(cls gain 0.5→1.5)** 앙상블에서 분류 정확도에 기여하는 체크포인트를 목표로 한다.
이 노트북 자체의 산출물은 **체크포인트**이며, Kaggle 제출용 CSV는 만들지 않는다.

| 항목 | 내용 |
|---|---|
| 학습 데이터 | 원본 train 232장(corrections 반영) + 합성 pool2 + **masked pool** — **task2-masked와 완전히 동일** (`syn_00505/00102` 제외 포함) |
| 클래스 | 74종 (train 56종 → 라벨 1~56, test 전용 18종 → 라벨 57~74) |
| fold 분할 | **재계산하지 않음** — task2-masked에서 export한 `fold_split_masked.json`을 Drive에서 로드. 세션/계정/라이브러리 버전 차이로 인한 분할 불일치를 원천 차단 (fold-matched WBF의 전제) |
| 모델/학습 | **YOLOv8m** (YOLO11 대비 가벼움 — masked pool 추가로 데이터가 커져 시간 절약), 100 epochs, imgsz 960, patience 15 |
| loss | box=7.5, **cls=1.5(기본 0.5의 3배)**, dfl=1.5 — Ultralytics 내장 loss gain 인자로 조정 (**loss 코드 수정 없음**) |
| test 추론 | 5-fold 앙상블 → WBF 융합 → 클래스별 crop으로 육안 확인 (제출 CSV 없음) |

**masked pool 처리 (task2-masked와 동일한 유의사항)**
- 파일명이 train과 동일 체계(K코드+촬영조건)라 원본과 겹칠 수 있음 → **전체 파일을 `msk_` 접두어로 리네임**한
  사본을 스테이징 폴더에 만들어 병합 (이미지 캐시 덮어쓰기·corrections 오적용 방지)
- annotation은 **원본 category_id를 그대로 사용** (합성 pool의 라벨 네임스페이스와 다름 — name 매핑 불필요)
- 같은 구성(위도/촬영조건만 다른 이미지 + masked 버전)의 train/valid 누수 방지 그룹화는
  `fold_split_masked.json`에 이미 반영되어 있으며, 로드 후 그룹 누수 assert로 재검증한다
- **masked pool 경로는 자동 탐색하지 않음** → 4번 셀의 `MASKED_ROOT`에 실제 Drive 경로를 직접 기입할 것


**실행 전제**
- Colab: **GPU 런타임** 필요 (Runtime → Change runtime type → GPU)
- Drive에 `fold_split_masked.json` 업로드 완료 상태여야 함 (4번 셀이 파일명으로 재귀 검색)
- `batch`는 실제 배정받은 GPU(T4/L4/A100 등) 메모리에 따라 OOM 시 낮춰야 한다.


In [ ]:
# [1. 환경 준비] ultralytics(YOLOv8 학습/추론 + COCO->YOLO 변환 유틸 포함), ensemble-boxes(WBF)
!pip install -q "ultralytics>=8.3" ensemble-boxes


In [ ]:
# [2. 저장소 준비] 팀 저장소 main 브랜치를 clone하고 RF_DETR_split_ver를 import 경로에 추가합니다.
#  ⚠ 저장소 파일/함수는 수정하지 않고 그대로 재사용합니다. (YOLO 관련 로직은 이 노트북에서 로컬로 정의)
import os, sys, re, glob, json, shutil
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
from PIL import Image

REPO_URL = 'https://github.com/kim-tae-yoon-0718/ai12-team01.git'
REPO_DIR = '/content/ai12-team01'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
sys.path.insert(0, os.path.join(REPO_DIR, 'RF_DETR_split_ver'))

# ── 저장소에서 그대로 재사용하는 함수 (경로 탐색 · 데이터 전처리 - 모델과 무관한 순수 로직) ──
from colab_setup import mount_drive
from dataset import (
    find_data_root,       # Drive에서 폴더 이름으로 데이터 루트 자동 탐색
    check_data_paths,     # train/test 이미지·annotation 하위 경로 존재 점검
    load_raw_annotations, # 박스당 1개로 흩어진 원본 annotation을 파일명 기준 병합 로드
    apply_corrections,    # corrections 기반 bbox 좌표/중복/누락/클래스 오기재 수정
    # make_folds는 일부러 import하지 않음 - fold 분할은 fold_split_masked.json 로드로 고정 (재계산 금지)
    cache_images,         # 이미지 로컬 캐시
    write_fold_dirs,      # fold별 {train,valid}에 COCO json + 이미지 배치
    save_label_map,       # label_map.json 저장
    compute_label_counts, # 클래스(라벨)별 박스 수 집계
)

# ── RF_DETR_split_ver 공용 모듈 - task2-masked/task4-kaggle과 반복되던 pool 병합/74종 매핑/
#    fold 로드/YOLO 학습/앙상블 후처리 로직을 모듈화한 것 (상세는 각 .py docstring 참고) ──
from pools import (
    load_pool_annotations, merge_pool,        # 합성 pool 로드(원본 id 공간 복원) + 병합
    merge_masked_pool,                          # masked pool 'msk_' 리네임 병합 (74종 매핑/이미지 존재 검증 포함)
    apply_exclusions, build_cat2label_74,       # 불량 이미지 제외 / 74종 라벨 매핑
)
from folds import (
    load_fold_split, assert_no_group_leak,      # 고정 분할 로드 + 그룹 누수 검증
    summarize_fold_distribution, print_fold_warnings,
)
from yolo import (
    get_yolo_model, build_yolo_fold,             # YOLOv8 모델 로드 / COCO->YOLO 변환
    train_fold_yolo, report_fold_result_yolo,    # fold 학습 + valid mAP 평가
    run_folds_yolo,                              # fold 목록을 받아 학습+리포팅 반복
    summarize_kfold_results_yolo, summarize_per_class_yolo,  # fold 평균/클래스별 mAP 집계
)
from ensemble import (
    fuse_predictions_wbf,             # WBF 융합
    collect_predictions_ensemble_yolo,  # test 폴더 fold 모델 합집합 추론
)
from visualize import show_pred_class_crops   # 클래스별 예측 crop grid 시각화 ('전부 표시' 모드)
from ultralytics import YOLO
from ultralytics.data.converter import convert_coco   # COCO json -> YOLO txt 공식 변환 유틸

CORRECTIONS_PATH = os.path.join(REPO_DIR, 'RF_DETR_split_ver', 'corrections.json')
print('corrections.json 존재:', os.path.exists(CORRECTIONS_PATH))
print('저장소 clone 완료:', REPO_DIR)


In [ ]:
# [4. Drive 마운트 + 경로 자동 탐색]
#  원본 train과 pool2가 모두 팀 공유 Drive 폴더 하위에 있다는 전제로, 폴더 "이름"으로 찾습니다.
#  팀원마다 상위 폴더 구조가 달라도(예: '1팀 공유 문서' 하위 여부) 동작하도록 후보 경로를 우선 탐색하고,
#  실패하면 search_root 아래를 재귀 검색합니다 (dataset.find_data_root 그대로 재사용).
mount_drive()

SEARCH_ROOT = '/content/drive/MyDrive'

DATA_ROOT = find_data_root(
    candidates=[
        '/content/drive/MyDrive',
        '/content/drive/MyDrive',
        '/content/drive/MyDrive',
    ],
    search_root=SEARCH_ROOT,
    target_name='sprint_ai_project1_data',
)
check_data_paths(DATA_ROOT)

POOL_ROOT = find_data_root(
    candidates=[
        '/content/drive/MyDrive/task2_synthesized',
        '/content/drive/MyDrive/task2_synthesized',
    ],
    search_root=SEARCH_ROOT,
    target_name='task2_synthesized',
)
POOL_IMG_DIR = os.path.join(POOL_ROOT, 'images')
POOL_ANN_PATH = (glob.glob(os.path.join(POOL_ROOT, 'annotations', '*.json')) or [None])[0]

print('DATA_ROOT:', DATA_ROOT)
print('POOL_ROOT:', POOL_ROOT)
print('POOL_IMG_DIR 존재:', os.path.exists(POOL_IMG_DIR) if POOL_IMG_DIR else False)
print('POOL_ANN_PATH:', POOL_ANN_PATH)
assert POOL_ANN_PATH, 'pool2 annotation json을 못 찾음 - 폴더 구조 확인 필요 (images/, annotations/*.json)'

# ── masked pool: 다른 위치에 있어 자동 탐색하지 않습니다. 실제 Drive 경로를 직접 기입하세요. ──
#    (폴더 안에 images/ 와 annotations/_annotations.coco.json 이 있어야 합니다)
MASKED_ROOT = '/content/drive/MyDrive/*** 여기에 masked pool(dataset-74-masked) 경로 기입 ***'   # TODO: 직접 기입
MASKED_IMG_DIR = os.path.join(MASKED_ROOT, 'images')
MASKED_ANN_PATH = (glob.glob(os.path.join(MASKED_ROOT, 'annotations', '*.json')) or [None])[0]
assert os.path.isdir(MASKED_IMG_DIR), (
    f'masked pool images 폴더 없음: {MASKED_IMG_DIR}\n-> 위 MASKED_ROOT를 실제 경로로 수정하세요')
assert MASKED_ANN_PATH, f"masked pool annotation json 없음: {os.path.join(MASKED_ROOT, 'annotations')}"

# ── 고정 fold 분할: task2-masked에서 export한 fold_split_masked.json을 Drive에서 파일명으로 검색 ──
_hits = glob.glob(os.path.join(SEARCH_ROOT, '**', 'fold_split_masked.json'), recursive=True)
assert _hits, f'fold_split_masked.json을 {SEARCH_ROOT} 아래에서 못 찾음 - 업로드 위치/파일명 확인'
if len(_hits) > 1:
    print(f'fold_split_masked.json 후보 {len(_hits)}개 -> 첫 번째 사용:\n  ' + '\n  '.join(_hits))
FOLD_SPLIT_JSON = _hits[0]

print('MASKED_ROOT:', MASKED_ROOT)
print('MASKED_ANN_PATH:', MASKED_ANN_PATH)
print('FOLD_SPLIT_JSON:', FOLD_SPLIT_JSON)


In [ ]:
# [5. 실험 설정]
#  task2-masked(RF-DETR medium, 100ep)와 동일한 데이터 조건에서 YOLOv8m을 학습합니다.
#  imgsz=960 유지: 원본 976x1280의 알약 디테일 보존 (32의 배수).
#  loss gain — Ultralytics 내장 하이퍼파라미터로 loss 성분별 가중치를 조절합니다 (loss 코드 무수정).
#   total = box_gain*box_loss + cls_gain*cls_loss + dfl_gain*dfl_loss  (v8DetectionLoss)
#   기본값 box=7.5, cls=0.5, dfl=1.5 (box 위주) -> cls를 0.5에서 1.5(3배)로 올려 분류 오류에
#   더 큰 패널티를 부여. WBF 앙상블에서 classification 정확도에 기여하는 체크포인트가 목적.
#   ⚠ 학습 후 results.png에서 box_loss 수렴이 눈에 띄게 나빠졌는지 확인할 것 (나빠졌으면 cls 하향 재조정).
TASK_ID = 4
TAG = 'yolov8m_task4_syn74_masked'

config = {
    'data': {
        'n_splits': 5,
        'seed': 42,          # (참고) fold 분할은 fold_split_masked.json 로드로 고정 - seed는 분할에 관여하지 않음
        'dataset_dir': '/content/dataset',             # COCO 포맷 fold 디렉토리 (write_fold_dirs 산출물)
        'cache_dir': '/content/img_cache',
        'yolo_dataset_dir': '/content/yolo_dataset',   # YOLO 포맷(images/labels) fold 디렉토리
    },
    'model': {
        'variant': 'medium',   # nano | small | medium | large | xlarge (yolov8 계열)
        'tag': TAG,
    },
    'train': {
        'epochs': 50,
        'imgsz': 960,
        'batch': 8,             # Colab GPU 메모리에 따라 조정 (OOM 시 4로 축소)
        'patience': 10,         # val 개선이 15ep 없으면 조기 종료 (데이터 증가에 따른 시간 절약)
        'seed': 42,
        'box_gain': 7.5,        # 기본값 유지
        'cls_gain': 1.5,        # 기본 0.5의 3배 - classification loss 패널티 강화
        'dfl_gain': 1.5,        # 기본값 유지
    },
    'output': {
        'local_output_dir': '/content/outputs',
        # 체크포인트/학습곡선을 Drive에 영구 저장 - 세션이 끊겨도 재마운트만 하면 이어하기 가능
        'backup_dir': '/content/drive/MyDrive',
    },
}

for d in (config['data']['cache_dir'], config['data']['dataset_dir'],
          config['data']['yolo_dataset_dir'], config['output']['local_output_dir'],
          config['output']['backup_dir']):
    os.makedirs(d, exist_ok=True)

VIS_SCORE_THR = 0.3   # test 예측 시각화용 confidence threshold (QA 목적 - 제출용 아님)
WBF_IOU_THR = 0.55

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU(!) - 런타임 유형 확인 필요')
print('backup_dir(영구 저장 - Drive):', config['output']['backup_dir'])
print('loss gains:', {k: v for k, v in config['train'].items() if k.endswith('_gain')})


In [ ]:
# [6. 원본 train 로드 + annotation 수정] Task2와 완전히 동일한 절차입니다.
boxes_by_image, cats_by_image, img_meta, ids_by_image = load_raw_annotations(
    os.path.join(DATA_ROOT, 'train_annotations'))
print(f"수정 전: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")

boxes_by_image, cats_by_image = apply_corrections(
    boxes_by_image, cats_by_image, ids_by_image, CORRECTIONS_PATH)
print(f"수정 후: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")

train_cats = sorted({c for cs in cats_by_image.values() for c in cs})
print('train 클래스 수:', len(train_cats))


In [ ]:
# [7. pool2 병합] task2-masked와 완전히 동일한 로직입니다 (pools.load_pool_annotations/merge_pool).
#  ⚠ fold 구성을 task2-masked와 일치시켜야 하므로, 병합 로직은 그대로 유지합니다.
pool_boxes, pool_cats, pool_ids, pool_meta, pool_coco = load_pool_annotations(POOL_ANN_PATH)
print(f"pool2: 이미지 {len(pool_meta)}장 / 박스 {sum(len(v) for v in pool_boxes.values())}개"
      f" / 클래스 {len({c for cs in pool_cats.values() for c in cs})}종")

merge_pool(boxes_by_image, cats_by_image, ids_by_image, img_meta,
          pool_boxes, pool_cats, pool_ids, pool_meta, pool_name='pool2')


In [ ]:
# [7-1. masked pool 병합] task2-masked(Kaggle) 노트북과 동일한 로직입니다 (pools.merge_masked_pool).
#   - masked pool의 _annotations.coco.json은 categories id가 "원본 category_id 그대로"라 name 매핑 없이 사용
#   - 파일명이 train과 동일 체계(K코드+촬영조건)라 원본과 겹칠 수 있어, 전체 파일을 'msk_' 접두어로
#     리네임한 사본을 스테이징 폴더에 생성 (cache_images 덮어쓰기·corrections 오적용 방지)
#   - 같은 구성(위도/촬영조건 변형 + masked 버전)의 split 누수 방지 그룹화는 fold_split_masked.json에
#     이미 반영되어 있고, 9번 셀에서 그룹 누수 assert로 재검증합니다
MASKED_STAGE_DIR = '/content/masked_renamed'
cats_74 = {int(c['name']) for c in pool_coco['categories'] if c['id'] != 0}
n_masked, masked_coco = merge_masked_pool(
    boxes_by_image, cats_by_image, ids_by_image, img_meta,
    MASKED_IMG_DIR, MASKED_ANN_PATH, MASKED_STAGE_DIR, cats_allowed=cats_74)


In [ ]:
# [7-2. 데이터 제외] task2-masked와 동일한 제외 목록을 반드시 그대로 적용합니다 (pools.apply_exclusions).
#  ⚠ fold_split_masked.json은 이 제외가 적용된 파일 집합 기준으로 만들어졌으므로,
#     목록이 다르면 9번 셀의 "분할-데이터 일치" assert에서 중단됩니다.
EXCLUDE_FILES = [
    'syn_00505.png',
    'syn_00102.png',
]
apply_exclusions(EXCLUDE_FILES, boxes_by_image, cats_by_image, ids_by_image, img_meta)


In [ ]:
# [8. 74종 라벨 매핑] Task2와 동일한 체계 (pools.build_cat2label_74): train 56종 -> 1~56,
#  test 전용 18종 -> 57~74. fold 구성을 Task2와 맞추려면 이 매핑도 완전히 동일해야 합니다.
cat2label, label2cat, all_cats = build_cat2label_74(pool_coco, train_cats, cats_by_image)


In [ ]:
# [9. 5-fold 분할 로드 + fold별 클래스 배분 점검] - 고정 분할 (재계산 없음)
#  StratifiedGroupKFold를 다시 돌리지 않고, task2-masked에서 export한 fold_split_masked.json을
#  folds.load_fold_split으로 로드합니다 (분할-데이터 불일치는 내부 assert로 검증). 이유:
#   1) fold-matched WBF: task2-masked RF-DETR 체크포인트와 fold별 valid 이미지 집합이 동일해야 함
#   2) 분할 재계산은 세션/계정 간 sklearn·numpy 버전 차이로 다른 분할이 나올 위험이 있음
file_names = sorted(boxes_by_image)
folds = load_fold_split(FOLD_SPLIT_JSON, file_names, config['data']['n_splits'])

# 그룹 누수 점검: 같은 그룹('msk_' 접두어 제거 기준 구성 코드)이 train/valid 양쪽에 있으면 안 됩니다.
assert_no_group_leak(folds, file_names)

fold_summary, valid_pivot = summarize_fold_distribution(folds, file_names, cats_by_image, cat2label)
display(fold_summary)

print_fold_warnings(fold_summary)

valid_pivot   # 라벨 x fold별 valid 박스 수 (셀 마지막 줄 -> 표로 표시. 클래스별 집계에서 재사용)


In [ ]:
# 이 셀 바로 위에 추가해서 확인
print("=== category mapping 확인 ===")
print(f"총 클래스 수: {len(cat2label)}")
print(f"\n{'category_id':>12} {'dl_idx':>10}")
print("-" * 25)
for name, cat_id in sorted(cat2label.items(), key=lambda x: x[1]):
    print(f"  {cat_id:>10} {name:>10}")

In [ ]:
# [10. COCO 포맷 fold 디렉토리 생성] Task2와 동일한 저장소 함수 조합입니다.
#  이 산출물(fold{i}/{train,valid}/_annotations.coco.json + 이미지)을 다음 셀에서 YOLO 포맷으로 변환합니다.
cache_images(os.path.join(DATA_ROOT, 'train_images'), config['data']['cache_dir'])
cache_images(POOL_IMG_DIR, config['data']['cache_dir'])
cache_images(MASKED_STAGE_DIR, config['data']['cache_dir'])   # masked 리네임 사본(msk_*.png)도 같은 캐시에 추가

write_fold_dirs(folds, file_names, boxes_by_image, cats_by_image, img_meta,
                cat2label, all_cats, config['data']['cache_dir'], config['data']['dataset_dir'])

# fold 디렉토리 복사가 끝나면 캐시는 더 필요 없으므로 삭제해 디스크를 확보합니다
#  (masked pool 추가로 데이터가 커져 fold 5개 사본 + 캐시가 공존하면 디스크가 부족할 수 있음)
shutil.rmtree(config['data']['cache_dir'], ignore_errors=True)
print('이미지 캐시 삭제 완료 (fold 디렉토리에 이미 복사됨)')

save_label_map(cat2label, label2cat, config['data']['dataset_dir'])
print('COCO 포맷 fold 디렉토리 생성 완료:', config['data']['dataset_dir'])


## 11. YOLO 포맷 변환

Ultralytics는 객체 검출 학습에 COCO json을 직접 쓰지 않고 `images/` + `labels/`(YOLO txt) 구조와
`data.yaml`을 요구한다 (확인 결과: 세그멘테이션/포즈가 아닌 detection 학습에 COCO json을 그대로 읽는
네이티브 경로는 없음). 다만 공식 변환 유틸 `ultralytics.data.converter.convert_coco()`가 이 변환을
대신해주므로 직접 파싱하지 않고 그대로 사용한다.

- `cls91to80=False`로 호출하면 `class_index = category_id - 1`을 그대로 쓴다 (COCO 80/91클래스
  리매핑을 켜지 않음 — 우리 데이터는 원래 COCO 80종이 아니므로 반드시 꺼야 함).
- 저장소 `build_coco()`가 넣는 더미 배경 카테고리(`id=0`, name='pill')에는 박스가 하나도 없으므로,
  별도로 제거하지 않아도 실제 74종이 정확히 YOLO class index `0~73`으로 매핑된다.
- `convert_coco()`는 라벨 txt만 생성하고 이미지는 복사하지 않으므로, 이미지는 심볼릭 링크로 연결해
  디스크 중복을 피한다.
- 변환된 txt 파일명은 COCO json의 `file_name`과 동일한 stem을 쓰므로, `images/`와 `labels/`를
  같은 파일명 규칙으로 나란히 두면 Ultralytics가 자동으로 짝을 찾는다.


In [ ]:
# [11. YOLO 포맷 변환 실행] (yolo.build_yolo_fold - COCO fold -> YOLO images/labels + data.yaml)
fold_yaml_paths = [
    build_yolo_fold(fi, config['data']['dataset_dir'], config['data']['yolo_dataset_dir'], label2cat)
    for fi in range(config['data']['n_splits'])
]
print('data.yaml 경로:')
for p in fold_yaml_paths:
    print(' ', p)


## 14. 5-fold 학습 실행

`run_kfold_yolo()` 하나로 fold마다 다음이 자동 수행됩니다.

1. `train_fold_yolo()` — 학습 후 best 체크포인트를 Drive `backup_dir`에 백업, `results.csv`/`results.png`도 함께 백업
2. `report_fold_result_yolo()` — 체크포인트를 다시 로드해 valid mAP 계산

**이어하기 (Colab 전용 장점)**: `backup_dir`가 Drive 경로이므로, 세션이 끊겨도 **런타임을 다시 시작하고
위 셀들을 순서대로 재실행한 뒤 이 셀만 다시 실행하면** 이미 완료된 fold는 자동으로 스킵됩니다
(Kaggle처럼 체크포인트를 Dataset으로 재업로드할 필요가 없음). fold 구성은 `fold_split_masked.json` 로드로
고정되어 세션/계정이 달라져도 동일하게 유지됩니다 (데이터 소스나 제외 목록이 바뀌면 9번 셀 assert에서
중단되므로 조용히 어긋날 수 없음).

⏱ **시간 주의**: medium × 100 epochs × 5 folds는 Colab 세션 한도를 넘길 수 있습니다.
처음에는 `run_kfold_yolo(config, fold_yaml_paths, max_folds=1)`로 fold 1개 소요 시간을 가늠해 보세요.


In [ ]:
# [14. 5-fold 학습 + 리포팅 실행] (yolo.run_folds_yolo - fold_indices=None이면 전체 fold 순회)
# 소요 시간 가늠용 sanity check: run_result = run_folds_yolo(config, fold_yaml_paths, max_folds=1)
run_result = run_folds_yolo(config, fold_yaml_paths)


In [ ]:
# [15. fold 결과 요약] Task2의 summarize_kfold_results/summarize_per_class와 같은 형태입니다
#  (yolo.summarize_kfold_results_yolo / summarize_per_class_yolo - Ultralytics box.maps가 이미
#  class index(0~73)로 고정 정렬되어 있어 torchmetrics 기반 버전보다 단순합니다).
kfold_summary = summarize_kfold_results_yolo(run_result['fold_metrics'], config['model']['tag'])

label_counts = compute_label_counts(config['data']['dataset_dir'])
per_class_df = summarize_per_class_yolo(run_result['fold_metrics'], label2cat, label_counts, valid_pivot)
per_class_df   # mean_AP 내림차순 - RF-DETR(Task2) 결과와 클래스별로 대조해볼 수 있음


## 16~18. test 5-fold 앙상블 추론 → WBF 융합 → 클래스별 예측 시각화

test에는 GT가 없으므로 mAP 계산 없이: 5개 fold 모델의 예측을 전부 수집(합집합) → **WBF**로 이미지
단위 융합 → 클래스별 crop 시각화(confidence 표시)로 육안 점검합니다. YOLO 성능을 눈으로 확인하는
것이 목적이라 Kaggle 제출용 CSV는 만들지 않습니다.


In [ ]:
# [16. test 추론 - 5 fold 합집합 수집] (ensemble.collect_predictions_ensemble_yolo)
#  Task2의 collect_predictions_ensemble()과 동일한 반환 스키마입니다
#  (rfdetr의 supervision.Detections 대신 ultralytics Results.boxes를 읽는 부분만 다름).
TEST_IMG_DIR = os.path.join(DATA_ROOT, 'test_images')
ckpts = [p for p in run_result['checkpoints'] if p]
assert len(ckpts) == config['data']['n_splits'], f"체크포인트 누락: {run_result['checkpoints']}"

test_pred_data = collect_predictions_ensemble_yolo(ckpts, TEST_IMG_DIR, conf_thr=0.05)
print('test 이미지 수:', len(test_pred_data))
print('합집합 예측 박스 수:', sum(len(d['pred_boxes']) for d in test_pred_data))


In [ ]:
# [17. WBF(Weighted Box Fusion) 융합] (ensemble.fuse_predictions_wbf - Task2 Kaggle 노트북과 동일 함수)
#  합집합 그대로면 같은 알약이 fold 수만큼 중복되어 육안 확인 시에도 같은 알약이 여러 번 겹쳐 보입니다.
fused_data = fuse_predictions_wbf(test_pred_data, n_models=len(ckpts),
                                  iou_thr=WBF_IOU_THR, skip_box_thr=0.05)
n_before = sum(len(d['pred_boxes']) for d in test_pred_data)
n_after = sum(len(d['pred_boxes']) for d in fused_data)
print(f'융합 전 박스 {n_before}개 -> 융합 후 {n_after}개')


In [ ]:
# [18. 클래스별 예측 시각화] (visualize.show_pred_class_crops - Task2 Kaggle 노트북과 동일 함수)
#  각 crop 제목: 1줄째 = 파일명(전체), 2줄째 = confidence score (클래스 내 score 내림차순 정렬)
#  ⚠ 한글 폰트 미지원 환경(Colab 기본 matplotlib) 대비, plot에 렌더링되는 텍스트는 영어로만 표기합니다.
# 특정 클래스만 보려면 classes=[57, 58] 처럼 '라벨' 번호로 지정하세요 (57~74 = test 전용 18종).
show_pred_class_crops(fused_data, label2cat, score_thr=VIS_SCORE_THR)


## 산출물

- `{backup_dir}/{tag}_fold{0..4}_best.pt` — 5-fold YOLOv8 체크포인트 (cls gain 1.5로 분류 강화 학습)
  (다음 단계에서 task2-masked RF-DETR와 fold-matched WBF 앙상블에 사용 — 두 실험이 같은
  `fold_split_masked.json`으로 fold를 나눴으므로 `fold{i}`끼리 valid 이미지 집합이 동일함)
- `{backup_dir}/{tag}_fold{i}_results.csv` / `_results.png` — fold별 학습 곡선 (Ultralytics 자동 생성)
- 이 노트북은 체크포인트 생성 + 육안 QA가 목적이라 Kaggle 제출용 CSV는 만들지 않습니다.
  최종 제출은 task2-masked(RF-DETR) + task4-masked(YOLOv8) 체크포인트를 함께 불러와 fold-matched WBF로
  앙상블하는 별도 노트북에서 진행합니다.
